In [ ]:
%pip install seaborn

In [ ]:
%pip install ydata-profiling

In [ ]:
import pandas as pd
import tkinter as tk
from tkinter import ttk


In [ ]:

import sys
import os

# Add src to the path
sys.path.append(os.path.abspath("../src"))
# Now you can import
from functions import *

In [ ]:
file_path = "C:/Users/bened/OneDrive/Desktop/Applied Statistics/Progetto/ANTHEM/data/processed_data/S3-approx-coordinates.parquet"

try:
    df = load_parquet_file(file_path)
    #preview_data(df)
    #visualize_data(df)

except Exception as e:
        print(f"Error: {e}")

In [ ]:
print("Column Names:")
print(df.columns.tolist())
print("\nNumber of Columns:", len(df.columns))
print("\nNumber of Rows:", len(df))

In [ ]:
from ydata_profiling import ProfileReport
# Create and generate the report
profile = ProfileReport(df, title="EDA Report", explorative=True)


In [ ]:
%pip install -U ipywidgets
import ipywidgets as widgets
#profile.to_notebook_iframe()

In [ ]:
# 3. Specify the exact columns you want to standardize
columns_to_standardize = ['CO2', 'P', 'PM1', 'PM25', 'RH', 'T', 'VOC', 'average_building_height_100', 'num_residential_buildings_200', 'num_commercial_buildings_200', 'num_trees_50', 'num_trees_500', 'num_green_200', 'num_green_500', 'num_green_1000', 'num_public_transport_50', 'num_public_transport_100', 'closeby_street_count_20', 'closeby_street_count_50', 'closeby_traffic_light_count_200', 'closeby_traffic_light_count_400', 'average_len_nearby_streets_50', 'average_nearby_maxspeed_50', 'average_nearby_num_lanes_50', 'len_nearby_residential_street_50', 'len_nearby_residential_street_150', 'len_nearby_pedestrian_street_15', 'len_nearby_pedestrian_street_50', 'len_nearby_service_street_50', 'len_nearby_service_street_150']

In [ ]:
from sklearn.preprocessing import StandardScaler

# 4. Apply standardization only to those columns
scaler = StandardScaler()
df_scaled = df.copy()
df_scaled[columns_to_standardize] = scaler.fit_transform(df[columns_to_standardize])

# 5. Display the profiling report after standardization
profile_after = ProfileReport(df_scaled, title="EDA Report - Selected Columns Standardized", explorative=True)
#profile_after.to_notebook_iframe()
profile_after.to_file("profile_after_report.html")

In [ ]:
from sklearn.feature_selection import VarianceThreshold

# Keep only numeric columns
X_numeric = df_scaled[columns_to_standardize]

# Apply VarianceThreshold
selector = VarianceThreshold(threshold=0.01)
X_var = selector.fit_transform(X_numeric)

# Optional: get selected column names
selected_var_cols = X_numeric.columns[selector.get_support()]
print("Selected features:", selected_var_cols.tolist())

In [ ]:
#Correlation filter
import numpy as np

corr_matrix = df_scaled[columns_to_standardize].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.75)]

X_corr = df_scaled.drop(columns=to_drop)
print("Dropped correlated features:", to_drop)

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression
X = df_scaled[columns_to_standardize]
X = X.dropna()
X_clean = X.drop(columns=['CO2'])
y_clean = X['CO2']

model = LinearRegression()
selector = RFE(estimator=model, n_features_to_select=20)
X_rfe = selector.fit_transform(X_clean, y_clean)

selected_rfe_cols = X_clean.columns[selector.get_support()]
print("Selected features by RFE:", selected_rfe_cols.tolist())

In [ ]:
from sklearn.linear_model import LassoCV

model = LassoCV()
model.fit(X_clean, y_clean)

selected_lasso_cols = X_clean.columns[model.coef_ != 0]
print("Selected features by Lasso:", selected_lasso_cols.tolist())

In [ ]:
%matplotlib inline
from sklearn.decomposition import PCA

pca = PCA(n_components=20)
X_pca = pca.fit_transform(X_clean)

explained_var = pca.explained_variance_ratio_
cumulative_var = np.cumsum(explained_var)

# 4. Plot explained variance
plt.figure(figsize=(12, 6))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(explained_var)+1), explained_var, alpha=0.6, color='blue', label='Individual')
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance per Component')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(cumulative_var)+1), cumulative_var, marker='o', linestyle='--', color='green', label='Cumulative')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance')
#plt.axhline(y=0.90, color='r', linestyle=':')  # Optional threshold line
plt.grid(True)

n_components_80 = np.argmax(cumulative_var >= 0.80) + 1
print(f"Number of components to retain 80% of variance: {n_components_80}")

# Get the loadings (components x features)
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(pca.components_))],
    index=X_clean.columns
)

# View top features per selected PC
for i in range(n_components_80):
    top_features = loadings.iloc[:, i].abs().sort_values(ascending=False).head(5)
    print(f"\nTop features for PC{i+1}:")
    print(top_features)

pc1_loadings = pd.Series(pca.components_[0], index=X_clean.select_dtypes(include=np.number).columns)

# Sort and plot
pc1_loadings.sort_values(ascending=False).plot(kind='bar', figsize=(12, 5), title="Feature Loadings on PC1")
plt.ylabel("Loading Magnitude")
plt.grid(True)
plt.show()